# Value Objects
A Value Object is an immutable object whose equality is based on its value, not identity.

Python's `dataclass(frozen=True)` is perfect for this.

## 1. The Problem Without Value Objects

In [ ]:
# Without value objects — just raw strings, no validation or guarantees
email = 'NOT_AN_EMAIL'  # nothing stops invalid data from being assigned
print(email)            # the program happily accepts garbage values

## 2. Value Object with dataclass

In [ ]:
import re
from dataclasses import dataclass

@dataclass(frozen=True)  # frozen=True makes this immutable — values cannot change after creation
class Email:
    value: str

    def __post_init__(self):  # runs automatically after __init__ — perfect for validation
        if not re.match(r'^[\w.-]+@[\w.-]+\.\w{2,}$', self.value):
            raise ValueError(f'Invalid email: {self.value}')  # reject invalid emails at creation
        # normalize to lowercase so 'Alice@Example.COM' == 'alice@example.com'
        object.__setattr__(self, 'value', self.value.lower())  # must use object.__setattr__ on frozen

    def __str__(self):
        return self.value  # print the email string directly


e1 = Email('Alice@Example.COM')  # normalized to lowercase in __post_init__
e2 = Email('alice@example.com')
print(e1)           # alice@example.com
print(e1 == e2)     # True — equality is based on value, not object identity

try:
    Email('bad-email')  # triggers validation in __post_init__
except ValueError as e:
    print(e)

## 3. More Value Objects

In [ ]:
@dataclass(frozen=True)  # immutable — a Money value should never change in place
class Money:
    amount: float
    currency: str = 'USD'  # default currency

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError('Amount cannot be negative')  # money can't be negative

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError('Currency mismatch')  # prevent adding USD to EUR
        return Money(self.amount + other.amount, self.currency)  # return a NEW Money object

    def __str__(self):
        return f'{self.currency} {self.amount:.2f}'  # format to 2 decimal places


m1 = Money(10.0)
m2 = Money(5.50)
print(m1 + m2)          # USD 15.50 — creates a new Money, doesn't modify m1 or m2
print(m1 == Money(10.0))  # True — same value means equal

## Exercise
1. Create a `PhoneNumber` value object with validation.
2. Create a `Percentage` value object that only accepts 0-100.
3. Create a `Coordinate` value object with lat/lon validation.